Mounting del Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Setup generale dell'ambiente

In [ ]:
# Installazione Ultralytics
!pip install -q ultralytics huggingface_hub

# Import
import os
import time
from pathlib import Path
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# Verifica dell'ambiente
print("=" * 50)
print("Setup ambiente")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 50)

Download del dataset da Roboflow

In [ ]:
!pip install -q roboflow pillow

import os
import yaml
import numpy as np
from pathlib import Path
from roboflow import Roboflow
from PIL import Image, ImageFile
from concurrent.futures import ThreadPoolExecutor

YOUR_API_KEY = ""    # Inserire qui la propria api_key di roboflow per fare il download del dataset
rf = Roboflow(api_key=YOUR_API_KEY)
project = rf.workspace("roboflow-universe-projects")\
             .project("license-plate-recognition-rxg4e")\
             .version(4)
dataset = project.download("yolov8")
print(f"Dataset scaricato in: {dataset.location}")

# Informazioni sul dataset
with open(f"{dataset.location}/data.yaml", "r") as f:
    data_info = yaml.safe_load(f)
print(f"Classi: {data_info['names']}")
print(f"Train: {data_info.get('train', 'N/A')}")
print(f"Val:   {data_info.get('val', 'N/A')}")

# Conversione del dataset in grayscale per il training
ImageFile.LOAD_TRUNCATED_IMAGES = True

corrotte = []

def converti_singola(img_path):
    try:
        img = Image.open(img_path).convert("L")
        img.save(img_path)
    except Exception as e:
        corrotte.append(img_path)
        img_path.unlink()
        label_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
        if label_path.exists():
            label_path.unlink()

def converti_grayscale(dataset_path):
    for split in ["train", "valid", "test"]:
        img_dir = Path(dataset_path) / split / "images"
        if not img_dir.exists():
            continue
        images = list(img_dir.glob("*.jpg")) + \
                 list(img_dir.glob("*.jpeg")) + \
                 list(img_dir.glob("*.png"))
        print(f"[{split}] Converto {len(images)} immagini...")
        with ThreadPoolExecutor(max_workers=8) as executor:
            executor.map(converti_singola, images)
        print(f"  [{split}] ✓ fatto")

converti_grayscale(dataset.location)
print("Conversione grayscale completata")
print(f"\nImmagini corrotte rimosse: {len(corrotte)}")
for p in corrotte:
    print(f"  - {p.name}")

# Si deve aggiungere il parametro channels: 1 nel data.yaml per poterle usare in grayscale
with open(f"{dataset.location}/data.yaml", "r") as f:
    data_info = yaml.safe_load(f)

data_info['channels'] = 1

with open(f"{dataset.location}/data.yaml", "w") as f:
    yaml.safe_dump(data_info, f, sort_keys=False)

print(f"Classi: {data_info['names']}")
print(f"Train: {data_info.get('train', 'N/A')}")
print(f"Val:   {data_info.get('val', 'N/A')}")
print(f"Channels: {data_info['channels']}")

Addestramento della rete YOLOv8n ridotta manualmente

In [ ]:
import ultralytics
import os
import shutil
from ultralytics import YOLO
import os
import glob
import shutil
import yaml
from pathlib import Path

src = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "models", "v8", "yolov8.yaml")
yaml_path = "/content/yolov8t.yaml"
shutil.copy(src, yaml_path)

# Viene aggiunta la scala 't' (tiny) riducendo la width a 0.125 (metà di yolov8n)
# Cambiato il numero dei canali con nc: 1
yaml_content = """
ch: 1
nc: 1
scales:
  t: [0.33, 0.125, 512]      # tiny: width 0.125 = metà di yolov8n

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"File scritto: {yaml_path}")

model = YOLO(yaml_path)

# Stampa dei parametri e stima della dimensione dei pesi
total_params = sum(p.numel() for p in model.model.parameters())
print(f"\nParametri totali:                {total_params:,}")
print(f"Dimensione pesi float32:         {total_params * 4 / (1024**2):.2f} MB")
print(f"Dimensione pesi INT8 (stimata):  {total_params / (1024**2):.2f} MB")

# Training YOLOv8t
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=192,
    batch=64,
    patience=15,
    device=0,
    name="yolov8t_lp_gray",
    lr0=0.01,
    pretrained=False,
)

print("\nTraining completato!")

Valutazione delle metriche e esportazione del modello in formato .onnx

In [ ]:
# Valutazione delle metriche
best_model = YOLO("/content/runs/detect/yolov8t_lp_gray/weights/best.pt")
metrics = best_model.val()
print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# Esportazione ONNX
best_model.export(format="onnx", imgsz=192, simplify=True)

Importazione di immagini da Google Drive per fare inferenza su immagini proprie

In [ ]:
from pathlib import Path
from PIL import Image, ImageFile
import shutil

ImageFile.LOAD_TRUNCATED_IMAGES = True

# Inserire qui il percorso alla cartella sul drive dove si contengono le immagini di targhe
drive_images = Path("")

test_dir = Path("/content/test_images")
test_dir.mkdir(exist_ok=True)

if not drive_images.exists():
    print(f"ERRORE: la cartella {drive_images} non esiste su Drive")
    print("Crea la cartella e carica li' le tue foto, poi rilancia")
else:
    valid_exts = ['.jpg', '.jpeg', '.png', '.webp']
    count = 0
    for f in drive_images.iterdir():
        if f.suffix.lower() in valid_exts:
            shutil.copy(f, test_dir / f.name)
            count += 1
            print(f"Copiato: {f.name} ({f.stat().st_size/1024:.1f} KB)")

    print(f"\nTotale immagini copiate: {count}")

    # Conversione delle immagini in grayscale (1 canale)
    print("\nConversione in grayscale (1 canale)...")
    for f in test_dir.iterdir():
        if f.suffix.lower() in valid_exts:
            try:
                img = Image.open(f).convert("L")
                img.save(f)
                print(f"  ✓ {f.name}: shape={img.size}, mode={img.mode}")
            except Exception as e:
                print(f"  ✗ {f.name}: errore - {e}")
                f.unlink()

# Elenco finale
print("\nImmagini disponibili per i test:")
for f in sorted(test_dir.iterdir()):
    if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name} ({size_kb:.1f} KB)")

Visualizzazione delle immagini di test

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

test_dir = Path("/content/test_images")
test_files = sorted([f for f in test_dir.iterdir()
                     if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']])

n = len(test_files)
print(f"Visualizzo {n} immagini di test\n")

cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
if n == 1:
    axes = [axes]
elif rows == 1:
    axes = list(axes)
else:
    axes = axes.flatten()

for ax, img_path in zip(axes, test_files):
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')   # cmap='gray' per immagini in grayscale
    ax.axis('off')
    ax.set_title(f"{img_path.name}\n{img.size[0]}x{img.size[1]}", fontsize=10)

for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

Inferenza del modello addestrato su tutte le immagini importate dal drive

In [ ]:
# Predict del modello
results = best_model.predict(
    source=str(test_dir),
    conf=0.5,
    imgsz=192,
    save=False,
    verbose=False
)

print(f"Processate {len(results)} immagini\n")
print("=" * 70)

# Stampa delle statistiche dettagliate per ogni immagine
for i, result in enumerate(results):
    img_name = Path(result.path).name
    img_w, img_h = result.orig_shape[1], result.orig_shape[0]
    n_plates = len(result.boxes)

    print(f"\n[{i+1}] {img_name} ({img_w}x{img_h})")

    if n_plates == 0:
        print("    Nessuna targa trovata (prova ad abbassare conf se ti aspetti detection)")
    else:
        for j, box in enumerate(result.boxes):
            conf = box.conf.item()
            xyxy = box.xyxy[0].tolist()
            w = xyxy[2] - xyxy[0]
            h = xyxy[3] - xyxy[1]
            # Area relativa: che frazione dell'immagine occupa la targa
            area_pct = (w * h) / (img_w * img_h) * 100

            print(f"    Targa {j+1}: confidenza = {conf:.1%}")
            print(f"       bbox (x,y,w,h) = ({xyxy[0]:.0f}, {xyxy[1]:.0f}, {w:.0f}, {h:.0f})")
            print(f"       area relativa: {area_pct:.2f}% dell'immagine")

print("\n" + "=" * 70)

# Riassunto finale
total_detections = sum(len(r.boxes) for r in results)
images_with_plates = sum(1 for r in results if len(r.boxes) > 0)
print(f"\nRiassunto:")
print(f"  Immagini totali: {len(results)}")
print(f"  Immagini con almeno una targa: {images_with_plates}/{len(results)}")
print(f"  Targhe totali rilevate: {total_detections}")

if total_detections > 0:
    avg_conf = sum(box.conf.item() for r in results for box in r.boxes) / total_detections
    print(f"  Confidenza media: {avg_conf:.1%}")

Visualizzazione delle immagini con il bounding box rilevato

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

n = len(results)
cols = min(2, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(9*cols, 6*rows))

# Normalizza axes come lista piatta
if n == 1:
    axes = [axes]
elif rows == 1:
    axes = list(axes)
else:
    axes = axes.flatten()

for ax, result in zip(axes, results):

    img_with_boxes = result.plot()
    img_with_boxes = img_with_boxes[..., ::-1]

    ax.imshow(img_with_boxes)
    ax.axis('off')

    img_name = Path(result.path).name
    n_plates = len(result.boxes)
    confs = [box.conf.item() for box in result.boxes]
    conf_str = ", ".join(f"{c:.1%}" for c in confs) if confs else "none"

    ax.set_title(f"{img_name} | {n_plates} detection | conf: {conf_str}", fontsize=11)

# Nascondi eventuali assi extra
for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\nInterpretazione:")
print("- Bounding box intorno alla targa = detection riuscita")
print("- Numero accanto al box = ID classe (0 = license_plate)")
print("- Percentuale = confidenza del modello")

Cella necessaria per splittare l'output tra bounding box e confidenza, altrimenti non funzionerà la dequantizzazione nella scheda STM32

In [ ]:
import onnx
from onnx import helper, TensorProto

model = onnx.load("") # Inserire qui il percorso al modello .onnx
graph = model.graph

# Verifica della input shape (deve essere [1,1,192,192])
input_tensor = graph.input[0]
print("Input shape attuale:", [d.dim_value for d in input_tensor.type.tensor_type.shape.dim])

# Rimozione del Concat finale (ultimo nodo)
last_concat = graph.node[-1]
assert last_concat.op_type == "Concat", "Ultimo nodo non è Concat!"
graph.node.remove(last_concat)

# Rimozione del vecchio output
while graph.output:
    graph.output.pop()

N_ANCHORS = 756  # Numero di anchor box per immagini 192x192

boxes_out = helper.make_tensor_value_info(
    '/model.22/Mul_2_output_0',
    TensorProto.FLOAT,
    [1, 4, N_ANCHORS]
)
conf_out = helper.make_tensor_value_info(
    '/model.22/Sigmoid_output_0',
    TensorProto.FLOAT,
    [1, 1, N_ANCHORS]
)

graph.output.extend([boxes_out, conf_out])

onnx.checker.check_model(model)
onnx.save(model, "yolov8t_lp_gray_split.onnx")
print("Salvato: yolov8t_lp_gray_split.onnx")
print(f"Output 1: boxes  [1, 4, {N_ANCHORS}]")
print(f"Output 2: conf   [1, 1, {N_ANCHORS}]")